# Fase 2: Auditoría y análisis exploratorio del dataset

En esta fase se realiza un análisis detallado del dataset de citas médicas con el objetivo de comprender su estructura, calidad y posibles problemas antes de entrenar modelos de Machine Learning.

## Contexto del dataset

El dataset utilizado corresponde a citas médicas y contiene información sobre pacientes, condiciones médicas y si asistieron o no a su cita.

La variable objetivo es:
- **No-show**: indica si el paciente NO asistió a su cita (Yes = no asistió, No = sí asistió)

In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016.csv"

df = pd.read_csv(path)

print("Dataset cargado correctamente")

Dataset cargado correctamente


In [3]:
db_path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\appointments.db"

conn = sqlite3.connect(db_path)

df.to_sql("appointments_raw", conn, if_exists="replace", index=False)

print("Dataset original guardado en SQL correctamente")

conn.close()

Dataset original guardado en SQL correctamente


In [4]:
conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT * FROM appointments_raw", conn)

conn.close()

print("Dataset cargado desde SQL")
print(df.shape)

Dataset cargado desde SQL
(110527, 14)


In [5]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Vista inicial del dataset

Se muestran las primeras filas para entender la estructura general de los datos.

In [6]:
df.shape

(110527, 14)

## Dimensiones del dataset

Aquí se observa el número de registros y columnas del dataset.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 11.8 MB


## Tipos de datos

Se analizan los tipos de datos de cada columna para identificar variables numéricas, categóricas y temporales.

In [8]:
df.isnull().sum()

PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

## Valores nulos

Se verifica si existen valores faltantes en el dataset.

In [9]:
df.duplicated().sum()

np.int64(0)

## Duplicados

Se identifican registros duplicados que podrían afectar el análisis.

In [10]:
df['No-show'].value_counts()

No-show
No     88208
Yes    22319
Name: count, dtype: int64

In [11]:
df['No-show'].value_counts(normalize=True)

No-show
No     0.798067
Yes    0.201933
Name: proportion, dtype: float64

## Distribución de la variable objetivo

Se analiza cuántos pacientes asistieron y cuántos no.

Este punto es crítico, ya que permite identificar si el dataset está desbalanceado.

In [12]:
df.describe()

,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


## Estadísticas descriptivas

Se analizan valores como media, desviación estándar, mínimos y máximos para variables numéricas.

In [13]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

PatientId: 62299
AppointmentID: 110527
Gender: 2
ScheduledDay: 103549
AppointmentDay: 27
Age: 104
Neighbourhood: 81
Scholarship: 2
Hipertension: 2
Diabetes: 2
Alcoholism: 2
Handcap: 5
SMS_received: 2
No-show: 2


## Hallazgos clave del dataset

A partir del análisis exploratorio realizado, se identificaron los siguientes puntos importantes:

- El dataset contiene 110,527 registros y 14 columnas, lo cual es suficiente para entrenar modelos de Machine Learning.
- No se encontraron valores nulos ni duplicados, lo que indica buena calidad estructural de los datos.
- La variable objetivo `No-show` presenta un desbalance significativo:
  - 79.81% de los pacientes asistieron
  - 20.19% no asistieron
- Este desbalance afecta directamente el rendimiento del modelo, especialmente en la detección de pacientes que no asisten.
- Se identificaron columnas que no aportan valor predictivo como `PatientId` y `AppointmentID`.
- Las variables `ScheduledDay` y `AppointmentDay` contienen información temporal importante, pero actualmente están en formato texto.
- Se detectaron anomalías en la variable `Age`, incluyendo valores negativos.
- La variable `Handcap` presenta múltiples valores, lo que sugiere que no es completamente binaria.

## Selección inicial de variables

Con base en el análisis, se clasifican las variables de la siguiente manera:

### Variables útiles (directas o transformables)
- Gender
- Age
- Neighbourhood
- Scholarship
- Hipertension
- Diabetes
- Alcoholism
- Handcap
- SMS_received
- ScheduledDay (requiere transformación)
- AppointmentDay (requiere transformación)

### Variables no útiles para el modelo
- PatientId (identificador)
- AppointmentID (identificador)

## Integración con base de datos SQL

El dataset original fue almacenado en una base de datos SQLite para simular un entorno más cercano a producción.

Posteriormente, los datos fueron consultados desde SQL y cargados en pandas para continuar con el flujo de análisis y modelado.

Esto permite desacoplar el proyecto de archivos estáticos y facilita su futura integración con sistemas reales.

## Conclusión

El análisis exploratorio permitió comprender la estructura, calidad y características del dataset.

Aunque los datos no presentan valores nulos ni duplicados, se identificaron aspectos críticos que deben ser atendidos en las siguientes fases, como el desbalance de clases, la presencia de valores atípicos y la falta de transformación en variables temporales.

Este análisis establece una base sólida para la fase de preprocesamiento, donde se limpiarán los datos y se generarán nuevas variables que mejoren la capacidad predictiva del modelo.